# Baseline models on char TF-IDF

Predict which LLM wrote a short text. Classical, sparse-friendly baselines only.

- TF-IDF: `analyzer="char"`, `ngram_range=(3, 8)`, `min_df=2`, `max_features=200_000`, `sublinear_tf=True`, `float32`. Fit **inside each fold**.
- CV: 5-fold `StratifiedGroupKFold` (groups = `prompt_id`), seed 69420.
- Metric: balanced accuracy. No class weighting anywhere.
- Sections: A `train.csv`, B `additional_train_1.csv`, C `additional_train_2.csv`, D all three joined.

In [ ]:
import os, random, time, warnings
SEED = 69420
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from pathlib import Path

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
N_JOBS = os.cpu_count()
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

## Data

In [ ]:
def find_data_file(name):
    # Search local repo, Kaggle and Colab; NTL_DATA_DIR overrides the search root
    roots = [os.environ.get("NTL_DATA_DIR"), "..", ".", "/kaggle/input", "/content"]
    skip = {".venv", ".git", "node_modules", "site-packages"}
    for root in filter(None, roots):
        root = Path(root)
        if not root.is_dir():
            continue
        for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
            dirnames[:] = [d for d in dirnames if d not in skip]
            if name in filenames:
                return Path(dirpath) / name
    raise FileNotFoundError(f"{name} not found; set NTL_DATA_DIR to the folder containing it")

def load_dataset(name):
    df = pd.read_csv(find_data_file(name))
    out = pd.DataFrame({
        "text": df["text"].astype(str),
        "model": df["model" if "model" in df else "label"],
        "group": name + ":" + df["prompt_id"].astype(str),
        "source": name,
    })
    return out

FILES = {
    "A": "train.csv",
    "B": "additional_train_1.csv",
    "C": "additional_train_2.csv",
}
DATASETS = {k: load_dataset(v) for k, v in FILES.items()}
DATASETS["D"] = pd.concat(DATASETS.values(), ignore_index=True)

for k, d in DATASETS.items():
    print(f"{k}: rows={len(d):,} classes={d['model'].nunique()} groups={d['group'].nunique()} "
          f"min/max per class={d['model'].value_counts().min()}/{d['model'].value_counts().max()}")

## CV harness

The vectorizer is refit on each training split. Each fold's matrices (CSR, float32) are built once and reused by every model, then freed (keeps memory flat on the joined set).

In [ ]:
def make_vectorizer():
    return TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 8),
        min_df=2,
        max_features=200_000,
        sublinear_tf=True,
        dtype=np.float32,
    )

def vectorize_fold(texts, tr, va):
    vec = make_vectorizer()
    Xtr = vec.fit_transform(texts[tr]).tocsr()
    Xva = vec.transform(texts[va]).tocsr()
    return Xtr, Xva

def make_splits(df):
    le = LabelEncoder()
    y = le.fit_transform(df["model"])
    groups = df["group"].to_numpy()
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    splits = list(cv.split(df["text"].to_numpy(), y, groups))
    for tr, va in splits:
        assert not set(groups[tr]) & set(groups[va]), "group leakage"
    return le, y, splits

## Models

All models take the sparse CSR matrix directly (no densifying). Notes:
- `Dummy` is the baseline (prior and stratified).
- `LGBM`: shallow trees, `zero_as_missing=False`, low `colsample_bytree` and `max_bin=63` to keep binning of 200k sparse columns cheap.
- `kNN` uses brute-force cosine on sparse rows; `NearestCentroid` runs on the L2-normalised TF-IDF rows.

In [ ]:
def model_zoo():
    return {
        "Dummy (prior)": DummyClassifier(strategy="prior"),
        "Dummy (stratified)": DummyClassifier(strategy="stratified", random_state=SEED),
        "MultinomialNB": MultinomialNB(),
        "ComplementNB": ComplementNB(),
        "NearestCentroid": NearestCentroid(),
        "kNN (cosine, k=5)": KNeighborsClassifier(n_neighbors=5, metric="cosine", algorithm="brute", n_jobs=N_JOBS),
        "DecisionTree (depth 10)": DecisionTreeClassifier(max_depth=10, random_state=SEED),
        "RidgeClassifier": RidgeClassifier(solver="sparse_cg", random_state=SEED),
        "SGD (modified_huber)": SGDClassifier(loss="modified_huber", max_iter=50, tol=1e-3, n_jobs=N_JOBS, random_state=SEED),
        "LinearSVC": LinearSVC(C=1.0, random_state=SEED),
        "LogisticRegression": LogisticRegression(solver="lbfgs", max_iter=200, random_state=SEED),
        "LGBM (depth 4)": LGBMClassifier(
            n_estimators=150, learning_rate=0.1, max_depth=4, num_leaves=15,
            colsample_bytree=0.2, min_child_samples=20, max_bin=63,
            zero_as_missing=False, force_col_wise=True, deterministic=True,
            n_jobs=N_JOBS, random_state=SEED, verbose=-1,
        ),
    }

def run_section(key):
    df = DATASETS[key]
    texts = df["text"].to_numpy()
    le, y, splits = make_splits(df)
    names = list(model_zoo())
    scores = {n: [] for n in names}
    fit_t = {n: 0.0 for n in names}
    oof = {n: np.full(len(df), -1) for n in names}
    for i, (tr, va) in enumerate(splits):
        t0 = time.time()
        Xtr, Xva = vectorize_fold(texts, tr, va)
        print(f"[{key}] fold {i}: vectorized in {time.time()-t0:.0f}s, features={Xtr.shape[1]:,}, nnz/doc={Xtr.nnz/Xtr.shape[0]:.0f}")
        for name, clf in model_zoo().items():
            t = time.time()
            clf.fit(Xtr, y[tr])
            fit_t[name] += time.time() - t
            p = clf.predict(Xva)
            oof[name][va] = p
            scores[name].append(balanced_accuracy_score(y[va], p))
        del Xtr, Xva
    rows = []
    for name in names:
        s = scores[name]
        rows.append({"model": name, "bal_acc_mean": np.mean(s), "bal_acc_std": np.std(s),
                     "fit_time_s": fit_t[name], **{f"fold{i}": v for i, v in enumerate(s)}})
    res = pd.DataFrame(rows).sort_values("bal_acc_mean", ascending=False).reset_index(drop=True)
    res.insert(0, "dataset", key)
    res.to_csv(RESULTS_DIR / f"baselines_{key}.csv", index=False)
    return res, oof, y, le

def plot_results(res, title):
    r = res.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.barh(r["model"], r["bal_acc_mean"], xerr=r["bal_acc_std"], color="#4c72b0")
    ax.set_xlabel("balanced accuracy (5-fold StratifiedGroupKFold)")
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout(); plt.show()

### Sanity check on a tiny subset
Checks that the harness has no group leakage and that the dummy baselines sit near chance (1/n_classes).

In [ ]:
_df = DATASETS["A"]
_le, _y, _splits = make_splits(_df)
print("classes:", len(_le.classes_), "-> chance balanced accuracy:", round(1 / len(_le.classes_), 4))
_tr, _va = _splits[0]
_Xtr, _Xva = vectorize_fold(_df["text"].to_numpy(), _tr, _va)
_p = DummyClassifier(strategy="stratified", random_state=SEED).fit(_Xtr, _y[_tr]).predict(_Xva)
print("stratified dummy, fold 0:", round(balanced_accuracy_score(_y[_va], _p), 4))
print("fold sizes (train/val):", [(len(a), len(b)) for a, b in _splits])
del _df, _le, _y, _splits, _Xtr, _Xva

## A. `train.csv`

In [ ]:
d = DATASETS["A"]
display(d["model"].value_counts().rename("n").to_frame().T)
res_A, oof_A, y_A, le_A = run_section("A")
display(res_A.drop(columns=["dataset"]).round(4))
plot_results(res_A, "train.csv")

## B. `additional_train_1.csv`

In [ ]:
d = DATASETS["B"]
display(d["model"].value_counts().rename("n").to_frame().T)
res_B, oof_B, y_B, le_B = run_section("B")
display(res_B.drop(columns=["dataset"]).round(4))
plot_results(res_B, "additional_train_1.csv")

## C. `additional_train_2.csv`

In [ ]:
d = DATASETS["C"]
display(d["model"].value_counts().rename("n").to_frame().T)
res_C, oof_C, y_C, le_C = run_section("C")
display(res_C.drop(columns=["dataset"]).round(4))
plot_results(res_C, "additional_train_2.csv")

## D. Joined train sets (A + B + C)

Labels unified to `model`; `prompt_id` is namespaced by file so groups never collide. Claude models only appear via A and B.

In [ ]:
d = DATASETS["D"]
display(d["model"].value_counts().rename("n").to_frame().T)
res_D, oof_D, y_D, le_D = run_section("D")
display(res_D.drop(columns=["dataset"]).round(4))
plot_results(res_D, "Joined train sets (A + B + C)")

### Joined set: out-of-fold balanced accuracy by source file

In [ ]:
src = DATASETS["D"]["source"].to_numpy()
rows = []
for name, pred in oof_D.items():
    r = {"model": name, "all": balanced_accuracy_score(y_D, pred)}
    for s in FILES.values():
        m = src == s
        r[s] = balanced_accuracy_score(y_D[m], pred[m])
    rows.append(r)
by_src = pd.DataFrame(rows).set_index("model").sort_values("all", ascending=False)
by_src.round(4)

## Summary: mean balanced accuracy, model x dataset

In [ ]:
allres = pd.concat([res_A, res_B, res_C, res_D])
summary = allres.pivot(index="model", columns="dataset", values="bal_acc_mean")
summary = summary[["A", "B", "C", "D"]].sort_values("D", ascending=False)
summary.columns = [f"{c}: {FILES.get(c, 'joined')}" for c in summary.columns]
allres.to_csv(RESULTS_DIR / "baselines_all.csv", index=False)
display(summary.round(4))
ax = summary.plot.bar(figsize=(11, 5), width=0.8)
ax.set_ylabel("balanced accuracy"); ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=40, ha="right"); plt.tight_layout(); plt.show()